In [1]:
!pip uninstall -y transformers sentence-transformers pillow pdfplumber

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4
Found existing installation: sentence-transformers 5.1.0
Uninstalling sentence-transformers-5.1.0:
  Successfully uninstalled sentence-transformers-5.1.0
Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
Found existing installation: pdfplumber 0.11.10
Uninstalling pdfplumber-0.11.10:
  Successfully uninstalled pdfplumber-0.11.10


In [2]:
!pip install --no-cache-dir \
transformers==4.55.4 \
sentence-transformers==5.1.0 \
pillow==12.3.0 \
pdfplumber==0.11.10 \
python-docx \
faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 6.4 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 6.3 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sentence-transformers]ence-transformers]


In [3]:
!pip install -q rapidfuzz

In [4]:
import pandas as pd
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import os
import re
import numpy as np
import pandas as pd
import torch
import faiss

import spacy
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from rapidfuzz import fuzz

/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

# DATA_PATH = BASE_PATH
# MODEL_PATH = BASE_PATH + "models/"
# OUTPUT_PATH = BASE_PATH + "outputs/"

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

DATA_PATH = BASE_PATH + "data/"
MODEL_PATH = BASE_PATH + "models/"
OUTPUT_PATH = BASE_PATH + "output/"

In [ ]:
job_df = pd.read_csv(DATA_PATH + "processed_jobs.csv")
index = faiss.read_index(MODEL_PATH + "job_faiss.index")
print("Job Data Shape :", job_df.shape)
print(index.ntotal)

Job Data Shape : (1615940, 15)
1615940


In [ ]:
resume_df = pd.read_csv(DATA_PATH + "processed_resume.csv")

In [ ]:
nlp = spacy.load("en_core_web_sm")

print("SpaCy Loaded Successfully")

SpaCy Loaded Successfully


In [ ]:
SKILL_SET = {

"python",
"java",
"c++",
"sql",
"excel",
"power bi",
"tableau",
"tensorflow",
"pytorch",
"keras",
"machine learning",
"deep learning",
"nlp",
"transformers",
"langchain",
"langgraph",
"scikit-learn",
"xgboost",
"lightgbm",
"docker",
"kubernetes",
"git",
"github",
"aws",
"azure",
"gcp",
"linux",
"statistics",
"data analysis",
"communication",
"leadership",
"project management",
"marketing",
"sales",
"human resources",
"recruitment",
"customer service"

}

In [ ]:
def extract_skills(text):

    text = str(text).lower()

    skills = set()

    for skill in SKILL_SET:

        if skill in text:
            skills.add(skill)

    return skills

In [ ]:
resume_text = resume_df.iloc[0]["resume_text"]

job_text = job_df.iloc[0]["job_text"]

resume_skills = extract_skills(resume_text)

job_skills = extract_skills(job_text)

print("Resume Skills:")
print(sorted(resume_skills))

print("\nJob Skills:")
print(sorted(job_skills))

Resume Skills:
['aws', 'customer service', 'data analysis', 'human resources', 'leadership', 'marketing', 'sales', 'statistics']

Job Skills:
['git', 'marketing']


In [ ]:
def compare_skills(resume_skills, job_skills):

    matched = resume_skills.intersection(job_skills)

    missing = job_skills - resume_skills

    if len(job_skills) == 0:
        score = 0
    else:
        score = len(matched) / len(job_skills) * 100

    return {
        "matched_skills": sorted(list(matched)),
        "missing_skills": sorted(list(missing)),
        "skill_match_score": round(score, 2)
    }

In [ ]:
result = compare_skills(
    resume_skills,
    job_skills
)

print(result)

{'matched_skills': ['marketing'], 'missing_skills': ['git'], 'skill_match_score': 50.0}


In [ ]:
def explain_match(resume_text, job_row):

    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_row["job_text"])

    comparison = compare_skills(
        resume_skills,
        job_skills
    )

    print("=" * 80)

    print("Job Title :", job_row["Job Title"])
    print("Company   :", job_row["Company"])

    print(f"\nCross Encoder Score : {job_row['cross_encoder_score']:.4f}")
    print(f"FAISS Score         : {job_row['faiss_score']:.4f}")

    print("\nMatched Skills")
    print(comparison["matched_skills"])

    print("\nMissing Skills")
    print(comparison["missing_skills"])

    print(f"\nSkill Match : {comparison['skill_match_score']:.2f}%")

    print("=" * 80)

In [ ]:
with open(MODEL_PATH + "embedding_model.txt") as f:
    MODEL_NAME = f.read().strip()

EMBEDDING_MODEL = SentenceTransformer(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
index = faiss.read_index(MODEL_PATH + "job_faiss.index")

In [ ]:
def get_resume_embedding(resume_text):

    embedding = EMBEDDING_MODEL.encode(
        resume_text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embedding.astype("float32")

In [ ]:
CROSS_ENCODER_PATH = MODEL_PATH + "resume_matching_model"

tokenizer = AutoTokenizer.from_pretrained(CROSS_ENCODER_PATH)

cross_encoder = AutoModelForSequenceClassification.from_pretrained(
    CROSS_ENCODER_PATH
)

In [ ]:
TOP_K = 50

def retrieve_jobs(resume_embedding):

    scores, indices = index.search(
        resume_embedding.reshape(1, -1),
        TOP_K
    )

    jobs = job_df.iloc[indices[0]].copy()

    jobs["faiss_score"] = scores[0]

    return jobs.reset_index(drop=True)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cross_encoder.to(device)
cross_encoder.eval()


def rerank_jobs(resume_text, jobs_df):

    scores = []

    with torch.no_grad():

        for job_text in jobs_df["job_text"]:

            inputs = tokenizer(
                resume_text,
                job_text,
                truncation=True,
                padding=True,
                max_length=512,
                return_tensors="pt"
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = cross_encoder(**inputs)

            probability = torch.softmax(
                outputs.logits,
                dim=1
            )[0][1].item()

            scores.append(probability)

    jobs_df = jobs_df.copy()

    jobs_df["cross_encoder_score"] = scores

    jobs_df = jobs_df.sort_values(
        "cross_encoder_score",
        ascending=False
    )

    return jobs_df.reset_index(drop=True)

In [ ]:
resume_df = pd.read_csv(DATA_PATH + "processed_resume.csv")

resume_text = resume_df.iloc[0]["resume_text"]

resume_embedding = get_resume_embedding(resume_text)

retrieved_jobs = retrieve_jobs(resume_embedding)

ranked_jobs = rerank_jobs(
    resume_text,
    retrieved_jobs
)

In [ ]:
def explain_match(resume_text, job_row):

    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_row["job_text"])

    comparison = compare_skills(
        resume_skills,
        job_skills
    )

    return {
        "Job Title": job_row["Job Title"],
        "Company": job_row["Company"],
        "Cross Encoder Score": job_row["cross_encoder_score"],
        "FAISS Score": job_row["faiss_score"],
        "Matched Skills": comparison["matched_skills"],
        "Missing Skills": comparison["missing_skills"],
        "Skill Match (%)": comparison["skill_match_score"]
    }

In [ ]:
reports = []

for _, job in ranked_jobs.iterrows():
    reports.append(explain_match(resume_text, job))

explanations = pd.DataFrame(reports)

display(explanations)

explanations.to_csv(
    OUTPUT_PATH + "job_match_explanations.csv",
    index=False
)

,Job Title,Company,Cross Encoder Score,FAISS Score,Matched Skills,Missing Skills,Skill Match (%)
0,Software Engineer,UnitedHealth Group,0.506855,0.787694,[],"[git, java]",0.00
1,Software Engineer,ConocoPhillips,0.506850,0.787694,[],"[git, java]",0.00
2,Graphic Designer,Xcel Energy,0.506828,0.788033,[],[git],0.00
3,UX/UI Designer,Vistra,0.506812,0.787440,[],[git],0.00
4,UX/UI Designer,Pfizer,0.506811,0.788527,[],[git],0.00
5,Systems Engineer,Tata Motors,0.506777,0.788527,[aws],"[azure, gcp]",33.33
6,Digital Marketing Specialist,Cholamandalam Investment and Finance Company,0.506773,0.788197,[marketing],[git],50.00
7,Software Tester,Kohl's,0.506772,0.788197,[],"[java, python]",0.00
8,Digital Marketing Specialist,Halliburton,0.506765,0.788527,[marketing],[git],50.00
9,Occupational Therapist,Lloyds Banking Group,0.506761,0.787694,[],[communication],0.00


In [ ]:
# for i in range(10):

#     resume_text = resume_df.iloc[i]["resume_text"]

#     resume_embedding = get_resume_embedding(resume_text)

#     retrieved_jobs = retrieve_jobs(resume_embedding)

#     ranked_jobs = rerank_jobs(
#         resume_text,
#         retrieved_jobs
#     )

#     print("=" * 100)
#     print(f"Resume {i+1}")
#     print(ranked_jobs[["Job Title", "Company", "cross_encoder_score"]].head(3))